# Multi-Emotion Caption Generator

## Research Design:
- **10 images** per run (API-friendly)
- **3 emotions** per image: joy, sad, surprised
- **4 techniques** per emotion: zero-shot, few-shot, chain-of-thought, persona
- **Total**: 10 x 3 x 4 = **120 captions per run**

In [1]:
import os
import pandas as pd
import google.generativeai as genai
from PIL import Image
import time
from tqdm import tqdm
import warnings
import json
from datetime import datetime
import gc
import random
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


c:\Users\Jeremy Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Jeremy Wijaya\AppData\Local\Temp\ipykernel_24684\2980420211.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import logging

# Configure logging without emoji to avoid Unicode errors
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler('multi_emotion_processing.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Change to project root
os.chdir('../../')
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\Jeremy Wijaya\Documents\KULIAH\Skripsi\Coding\IMAGE-CAPTIONING


In [3]:
# CONFIGURATION - CHANGE THESE FOR EACH RUN
# Load API key from .env file (secure method)
load_dotenv()
API_KEY = os.getenv('GEMINI_API_KEY')

if not API_KEY:
    raise ValueError("API key not found! Please set GEMINI_API_KEY in .env file")

CONFIG = {
    # File paths
    'csv_input': "data/raw/filenames_with_mood.csv",
    'folder_gambar': "scaled_images",
    'output_file': "data/multi_emotion_run_new2.csv",  # CHANGE: run1, run2, run3, run4
    'log_file': "data/multi_emotion_run_new2.json",
    'selection_file': "data/multi_emotion_selection_run_new2.json",
    
    # Research settings
    'total_images': 20,  # 10 images per run
    'emotions_per_image': 3,
    'techniques_per_emotion': 4,
    'total_captions': 120,  # 10 x 3 x 4
    'random_seed': 42,  # CHANGE: 42, 43, 44, 45 for each run
    
    # API settings
    'base_delay': 2.0,
    'max_delay': 20.0,
    'retry_attempts': 3,
    'max_image_size': (1024, 1024),
    'gc_interval': 5,
    
    # Emotions and techniques
    'emotions': ['joy', 'sad', 'surprised'],
    'prompting_techniques': ['zero-shot', 'few-shot', 'chain-of-thought', 'persona']
}

# Initialize Gemini
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

print("Configuration completed!")
print(f"Images: {CONFIG['total_images']}, Captions: {CONFIG['total_captions']}")
print(f"Output: {CONFIG['output_file']}")

Configuration completed!
Images: 20, Captions: 120
Output: data/multi_emotion_run_new2.csv


In [4]:
def get_prompt(emotion, technique):
    """Get prompt for specific emotion and technique"""
    prompts = {
        "zero-shot": {
            "joy": """Create ONE short English caption for this image with a joyful and cheerful mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "sad": """Create ONE short English caption for this image with a sad and melancholic mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "surprised": """Create ONE short English caption for this image with a surprised and amazed mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain."""
        },
        "few-shot": {
            "joy": """Follow these examples:
Joyful: 'What an amazing day to start a new adventure!'
Sad: 'Sometimes silence is the best companion for reflection.'
Surprised: 'Wow, this beauty is truly unexpected!'

Now create ONE English caption for joyful mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "sad": """Follow these examples:
Joyful: 'What an amazing day to start a new adventure!'
Sad: 'Sometimes silence is the best companion for reflection.'
Surprised: 'Wow, this beauty is truly unexpected!'

Now create ONE English caption for sad mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "surprised": """Follow these examples:
Joyful: 'What an amazing day to start a new adventure!'
Sad: 'Sometimes silence is the best companion for reflection.'
Surprised: 'Wow, this beauty is truly unexpected!'

Now create ONE English caption for surprised mood.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain."""
        },
        "chain-of-thought": {
            "joy": """Analyze this image internally.
Then create ONE final short English caption with a cheerful and joyful mood.
Return ONLY the final caption.
Do not explain your reasoning.
Do not provide alternatives.""",
            "sad": """Analyze this image internally.
Then create ONE final short English caption with a sad and melancholic mood.
Return ONLY the final caption.
Do not explain your reasoning.
Do not provide alternatives.""",
            "surprised": """Analyze this image internally.
Then create ONE final short English caption with a surprised and amazed mood.
Return ONLY the final caption.
Do not explain your reasoning.
Do not provide alternatives."""
        },
        "persona": {
            "joy": """You are an Influencer Specialist expert in audience psychology.
Create ONE highly engaging English caption for this image with joyful impression.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "sad": """You are an Influencer Specialist expert in audience psychology.
Create ONE highly engaging English caption for this image with sad impression.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain.""",
            "surprised": """You are an Influencer Specialist expert in audience psychology.
Create ONE highly engaging English caption for this image with surprised impression.
Return ONLY one sentence.
Do not provide alternatives.
Do not explain."""
        }
    }
    
    return prompts.get(technique, {}).get(emotion, prompts["zero-shot"]["joy"])

print("Prompting functions defined!")

Prompting functions defined!


In [5]:
def select_images(df_input, config):
    """Select random images for processing (with duplicate prevention)"""
    
    # Get available images
    available = []
    for _, row in df_input.iterrows():
        path = os.path.join(config['folder_gambar'], row['filename'])
        if os.path.exists(path):
            available.append(row['filename'])
    
    print(f"Available images: {len(available)}")
    
    if len(available) == 0:
        print("ERROR: No images found in folder!")
        print(f"Please add images to: {config['folder_gambar']}")
        return []
    
    # ANTI-DUPLICATE: Check previous runs
    previously_selected = []
    for run_num in range(1, 5):  # Check run1, run2, run3, run4
        prev_file = f"data/multi_emotion_selection_run{run_num}.json"
        if os.path.exists(prev_file):
            try:
                with open(prev_file, 'r') as f:
                    prev_data = json.load(f)
                    previously_selected.extend(prev_data.get('selected_images', []))
            except:
                pass
    
    if previously_selected:
        print(f"Previously selected: {len(set(previously_selected))} images")
        # Remove duplicates
        available = [img for img in available if img not in previously_selected]
        print(f"Available after filtering: {len(available)} images")
    
    if len(available) < config['total_images']:
        print(f"WARNING: Only {len(available)} images available")
        config['total_images'] = len(available)
        config['total_captions'] = config['total_images'] * 3 * 4
    
    # Shuffle and select
    random.shuffle(available)
    selected = available[:config['total_images']]
    
    print(f"Selected: {len(selected)} NEW images")
    
    # Save selection
    selection = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'random_seed': config['random_seed'],
        'total_images': len(selected),
        'selected_images': selected
    }
    
    with open(config['selection_file'], 'w') as f:
        json.dump(selection, f, indent=2)
    
    return selected

print("Image selection function defined!")

Image selection function defined!


In [6]:
class CaptionProcessor:
    def __init__(self, config):
        self.config = config
        self.model = model
        self.delay = config['base_delay']
        self.success = 0
        self.error = 0
        self.start = time.time()
    
    def load_image(self, path):
        try:
            img = Image.open(path)
            if img.size[0] > 1024 or img.size[1] > 1024:
                img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            return img
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None
    
    def generate(self, img, emotion, technique, filename):
        prompt = get_prompt(emotion, technique)
        
        for attempt in range(self.config['retry_attempts']):
            try:
                response = self.model.generate_content([prompt, img])
                result = response.text.strip()
                
                if result.startswith('"') and result.endswith('"'):
                    result = result[1:-1]
                
                print(f"  OK: {technique}-{emotion}")
                self.success += 1
                time.sleep(self.delay)
                return result
                
            except Exception as e:
                print(f"  FAIL: {technique}-{emotion} (attempt {attempt+1}): {e}")
                if attempt < self.config['retry_attempts'] - 1:
                    time.sleep((2 ** attempt) * self.delay)
                else:
                    self.error += 1
                    return f"Error: {str(e)[:50]}"
    
    def process_image(self, filename):
        """Process one image for all emotions and techniques"""
        results = []
        path = os.path.join(self.config['folder_gambar'], filename)
        
        print(f"\nProcessing: {filename}")
        
        if not os.path.exists(path):
            print(f"  ERROR: File not found")
            return []
        
        img = self.load_image(path)
        if img is None:
            return []
        
        # Process each emotion
        for emotion in self.config['emotions']:
            print(f"  Emotion: {emotion}")
            
            # Process each technique
            for technique in self.config['prompting_techniques']:
                caption = self.generate(img, emotion, technique, filename)
                
                results.append({
                    'filename': filename,
                    'emotion': emotion,
                    'technique': technique,
                    'caption': caption,
                    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'success': not caption.startswith('Error')
                })
        
        img.close()
        return results
    
    def get_stats(self):
        elapsed = time.time() - self.start
        total = self.success + self.error
        return {
            'total': total,
            'success': self.success,
            'error': self.error,
            'rate': self.success / max(total, 1),
            'time': elapsed
        }

print("Processor class defined!")

Processor class defined!


In [7]:
print("="*60)
print("LOADING DATASET")
print("="*60)

# Load CSV
df_input = pd.read_csv(CONFIG['csv_input'])
print(f"Dataset: {len(df_input)} rows")
print(f"Columns: {df_input.columns.tolist()}")

# Check images folder
if os.path.exists(CONFIG['folder_gambar']):
    images = [f for f in os.listdir(CONFIG['folder_gambar']) 
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"\nImages in folder: {len(images)}")
    if len(images) > 0:
        print(f"Sample: {images[:20]}")
    else:
        print("WARNING: No images found!")
        print(f"Please add images to: {CONFIG['folder_gambar']}")
else:
    print(f"\nERROR: Folder not found: {CONFIG['folder_gambar']}")

print("="*60)

LOADING DATASET
Dataset: 31783 rows
Columns: ['filename', 'mood_1', 'mood_2', 'mood_3']

Images in folder: 20
Sample: ['2209751.jpg', '2285664.jpg', '2317271.jpg', '2656351.jpg', '2689611.jpg', '2760167.jpg', '2784746.jpg', '2806447.jpg', '2868798.jpg', '3001353.jpg', '3012229.jpg', '3025093.jpg', '3035057.jpg', '3043766.jpg', '3160699.jpg', '3219606.jpg', '3367399.jpg', '3494059.jpg', '3537322.jpg', '3637013.jpg']


In [8]:
# EXECUTE PROCESSING
# Run this cell to start generating captions

print("="*60)
print("STARTING PROCESSING")
print("="*60)

# Initialize
processor = CaptionProcessor(CONFIG)
selected = select_images(df_input, CONFIG)

if len(selected) == 0:
    print("\nERROR: No images to process!")
    print("Please check:")
    print(f"1. Folder exists: {CONFIG['folder_gambar']}")
    print(f"2. Images are in folder")
    print(f"3. CSV file exists: {CONFIG['csv_input']}")
else:
    all_results = []
    
    print(f"\nProcessing {len(selected)} images...")
    print(f"Each image: 3 emotions x 4 techniques = 12 captions")
    print(f"Total: {len(selected) * 12} captions\n")
    
    # Process each image
    for i, filename in enumerate(selected):
        print(f"\n[{i+1}/{len(selected)}] {filename}")
        
        results = processor.process_image(filename)
        all_results.extend(results)
        
        # Save progress every 5 images
        if (i + 1) % 5 == 0:
            df_temp = pd.DataFrame(all_results)
            df_temp.to_csv(CONFIG['output_file'], index=False)
            print(f"\n  Progress saved: {len(all_results)} captions")
        
        # Garbage collection
        if (i + 1) % CONFIG['gc_interval'] == 0:
            gc.collect()
    
    # Final save
    df_final = pd.DataFrame(all_results)
    df_final.to_csv(CONFIG['output_file'], index=False)
    
    # Save stats
    stats = processor.get_stats()
    with open(CONFIG['log_file'], 'w') as f:
        json.dump(stats, f, indent=2)
    
    # Summary
    print("\n" + "="*60)
    print("PROCESSING COMPLETED!")
    print("="*60)
    print(f"Total captions: {len(all_results)}")
    print(f"Success: {stats['success']}")
    print(f"Errors: {stats['error']}")
    print(f"Success rate: {stats['rate']:.1%}")
    print(f"Time: {stats['time']/60:.1f} minutes")
    print(f"\nSaved to: {CONFIG['output_file']}")
    print("="*60)

STARTING PROCESSING
Available images: 20
Selected: 20 NEW images

Processing 20 images...
Each image: 3 emotions x 4 techniques = 12 captions
Total: 240 captions


[1/20] 3637013.jpg

Processing: 3637013.jpg
  Emotion: joy
  OK: zero-shot-joy
  OK: few-shot-joy
  OK: chain-of-thought-joy
  OK: persona-joy
  Emotion: sad
  OK: zero-shot-sad
  OK: few-shot-sad
  OK: chain-of-thought-sad
  OK: persona-sad
  Emotion: surprised
  OK: zero-shot-surprised
  OK: few-shot-surprised
  OK: chain-of-thought-surprised
  OK: persona-surprised

[2/20] 2760167.jpg

Processing: 2760167.jpg
  Emotion: joy
  OK: zero-shot-joy
  OK: few-shot-joy
  OK: chain-of-thought-joy
  OK: persona-joy
  Emotion: sad
  OK: zero-shot-sad
  OK: few-shot-sad
  OK: chain-of-thought-sad
  OK: persona-sad
  Emotion: surprised
  OK: zero-shot-surprised
  OK: few-shot-surprised
  OK: chain-of-thought-surprised
  OK: persona-surprised

[3/20] 3160699.jpg

Processing: 3160699.jpg
  Emotion: joy
  OK: zero-shot-joy
  OK: few-sho

In [ ]:
# VIEW RESULTS
# Run this cell after processing to see results

try:
    df = pd.read_csv(CONFIG['output_file'])
    
    print("="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(f"Total rows: {len(df)}")
    print(f"Unique images: {df['filename'].nunique()}")
    print(f"\nEmotions:")
    print(df['emotion'].value_counts())
    print(f"\nTechniques:")
    print(df['technique'].value_counts())
    print(f"\nSuccess rate: {df['success'].mean():.1%}")
    
    print("\n" + "="*60)
    print("SAMPLE RESULTS (first image)")
    print("="*60)
    first = df['filename'].iloc[0]
    sample = df[df['filename'] == first]
    print(sample[['emotion', 'technique', 'caption', 'success']])
    
except FileNotFoundError:
    print("No results file found yet. Run processing first!")